[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsobaCloud/odse/blob/main/colab/thokoza_level2_detector_colab_demo.ipynb)

# Thokoza Level 2 Detector Demo — ODS-E Physical Residuals (SEP-053)

Run **zero-GPU Level 2 anomaly detection** on real feeder-site telemetry from **Thokoza Logistics Park** (Durban 11 kV feeder, **1.2 MW PV**, 4 Huawei string inverters + site BESS).

This notebook ships with ODS-E **0.12.0** (`odse.physics` + `odse.detectors`):

1. Join 5-min inverter SCADA to hourly Durban weather ambient
2. Inject deterministic physics residuals (`inject_physical_residuals`)
3. Run `BinaryDropoutDetector`, `ThermalDivergenceDetector`, and `PhysicalHeuristicDetector`
4. Triage midday hardware-style dropouts without deep learning

**Datasets** (GitHub raw — no clone required):

| File | Contents |
|---|---|
| `thokoza_telemetry_6mo.csv.gz` | 5-min telemetry × 4 PV inverters + BESS (Mar–Sep 2026) |
| `durban_weather_6mo.csv.gz` | Hourly ambient for Faiman cell temperature |

> Sibling site to the Durban Feeder Grid demo — **not** the Mzansi textile plant.


In [ ]:
# ODS-E 0.12.0+ required for odse.physics / odse.detectors (SEP-053)
!pip install -q "odse[detectors]>=0.12.0" matplotlib


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from odse import __version__
from odse.adapters.pvlib import inject_physical_residuals
from odse.detectors import (
    BinaryDropoutDetector,
    PhysicalHeuristicDetector,
    ThermalDivergenceDetector,
)
from odse.validator import validate_batch

print(f"odse {__version__}")

BASE = "https://raw.githubusercontent.com/AsobaCloud/odse/main/colab/data"
RATED_KW = 240.0  # ~1.2 MW / 4 string inverters
HW_FAULT_STATES = {768, 770, 771, 45056}


In [ ]:
# Load Thokoza PV inverters only (exclude BESS channel for residual physics)
try:
    tel = pd.read_csv(f"{BASE}/thokoza_telemetry_6mo.csv.gz")
    weather = pd.read_csv(f"{BASE}/durban_weather_6mo.csv.gz")
except Exception:
    tel = pd.read_csv("colab/data/thokoza_telemetry_6mo.csv.gz")
    weather = pd.read_csv("colab/data/durban_weather_6mo.csv.gz")

pv = tel[tel["asset_id"].astype(str).str.startswith("INV-")].copy()
print(f"Thokoza rows: {len(tel):,} | PV inverter rows: {len(pv):,}")
print(f"Assets: {sorted(pv['asset_id'].unique())}")
print(pv[["timestamp", "asset_id", "power", "irradiance", "temperature", "error_type"]].head())


In [ ]:
# Validate a sample batch against the ODS-E schema
sample = pv.head(40)
records = []
for r in sample.itertuples():
    records.append({
        "timestamp": str(r.timestamp),
        "kWh": float(r.kwh),
        "kW": float(r.power),
        "voltage_ac": float(r.voltage),
        "temperature": float(r.temperature),
        "poa_irradiance": float(r.irradiance),
        "wind_speed_ms": float(r.wind_speed),
        "error_type": str(r.error_type),
        "asset_id": str(r.asset_id),
    })
result = validate_batch(records)
print(f"validate_batch: valid={getattr(result, 'is_valid', None)} "
      f"passed={getattr(result, 'passed', None)} failed={getattr(result, 'failed', None)}")


## 1. Ambient join + physical residuals

`inject_physical_residuals` requires ambient air temperature (no silent night-proxy).
We asof-merge hourly Durban `temp` onto 5-min SCADA, then compute Faiman $T_{cell}$,
$P_{expected}$, $\Delta P_{norm}$, and $\Delta T_{divergence}$.


In [ ]:
pv["_ts"] = pd.to_datetime(pv["timestamp"], utc=True)
weather = weather.copy()
weather["_ts"] = pd.to_datetime(weather["timestamp"], utc=True)
weather = weather.sort_values("_ts")
pv = pv.sort_values("_ts")

merged = pd.merge_asof(
    pv,
    weather[["_ts", "temp"]].rename(columns={"temp": "ambient_temperature_c"}),
    on="_ts",
    direction="backward",
)
merged["ambient_temperature_c"] = merged["ambient_temperature_c"].bfill().ffill()
merged = merged.drop(columns=["_ts"])

enriched = inject_physical_residuals(merged, rated_capacity_kw=RATED_KW, temp_coeff=-0.0035)
print(enriched[["power", "irradiance", "p_expected", "delta_p_norm", "delta_t_divergence"]].describe().round(3))


## 2. Level 2 detectors

- **BinaryDropoutDetector** — midday generation collapse ($G > 200$, $\Delta P_{norm} < -0.50$)
- **ThermalDivergenceDetector** — heatsink vs Faiman cell divergence sustained ≥ 3 steps
- **PhysicalHeuristicDetector** — composite event stream for ops triage


In [ ]:
dropout = BinaryDropoutDetector(tau=0.50, g_min=200.0)
thermal = ThermalDivergenceDetector(tau=15.0, min_consecutive=3)
composite = PhysicalHeuristicDetector()

dropout_events = dropout.detect(enriched)
thermal_events = thermal.detect(enriched)
all_events = composite.detect(enriched)

print(f"Binary dropout events: {len(dropout_events):,}")
print(f"Thermal divergence events: {len(thermal_events):,}")
print(f"Composite events: {len(all_events):,}")
print(all_events["detector_type"].value_counts() if len(all_events) else "no events")
all_events.head(10)


In [ ]:
# Daytime residual AUROC vs OEM hardware-fault state codes (benchmark style)
y_true = enriched["inverter_state"].isin(HW_FAULT_STATES).astype(int).to_numpy()
daytime = (enriched["irradiance"] > 200.0).to_numpy()
scores = np.where(daytime, -enriched["delta_p_norm"].to_numpy(), 0.0)

pos = scores[daytime & (y_true == 1)]
neg = scores[daytime & (y_true == 0)]
n_pos, n_neg = len(pos), len(neg)
if n_pos and n_neg:
    order = np.argsort(scores[daytime])
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, daytime.sum() + 1, dtype=float)
    y_day = y_true[daytime]
    sum_pos = ranks[y_day == 1].sum()
    auroc = (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    print(f"Daytime BinaryDropout residual AUROC: {auroc:.4f}  (n_pos={n_pos}, n_neg={n_neg})")
else:
    print("Insufficient daytime fault labels for AUROC")


## 3. Visual triage — one inverter week with a dropout


In [ ]:
inv = "INV-THK-01"
one = enriched[enriched["asset_id"] == inv].copy()
one["timestamp"] = pd.to_datetime(one["timestamp"], utc=True)

# Prefer a day that contains at least one binary-dropout flag
flags = dropout.score(one) > 0
if flags.any():
    hit_day = one.loc[flags, "timestamp"].iloc[0].floor("D")
else:
    hit_day = one["timestamp"].median().floor("D")

window = one[(one["timestamp"] >= hit_day) & (one["timestamp"] < hit_day + pd.Timedelta(days=1))]

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(window["timestamp"], window["irradiance"], color="#f59e0b", label="POA irradiance")
axes[0].set_ylabel("W/m²")
axes[0].legend(loc="upper right")
axes[0].set_title(f"Thokoza {inv} — {hit_day.date()} physics residuals & Level 2 flags")

axes[1].plot(window["timestamp"], window["p_expected"], color="#64748b", label="P_expected", lw=1.2)
axes[1].plot(window["timestamp"], window["power"], color="#455BF1", label="P_ac measured", lw=1.4)
axes[1].set_ylabel("kW")
axes[1].legend(loc="upper right")

axes[2].plot(window["timestamp"], window["delta_p_norm"], color="#0f172a", label="ΔP_norm")
axes[2].axhline(-0.50, color="#dc2626", ls="--", lw=1, label="dropout τ = −0.50")
hit_mask = (window["irradiance"] > 200) & (window["delta_p_norm"] < -0.50)
if hit_mask.any():
    axes[2].scatter(
        window.loc[hit_mask, "timestamp"],
        window.loc[hit_mask, "delta_p_norm"],
        color="#dc2626", s=28, zorder=5, label="BinaryDropout flag",
    )
axes[2].set_ylabel("ΔP_norm")
axes[2].legend(loc="lower right")
axes[2].set_xlabel("UTC")
plt.tight_layout()
plt.show()


## 4. Ops takeaway

Level 2 residuals catch **obvious midday power collapse** on CPU in milliseconds — the same class of contactor / thermal lockout faults that dominate utility SCADA — without shipping a multi-GB neural model to the edge.

```python
from odse.adapters.pvlib import inject_physical_residuals
from odse.detectors import PhysicalHeuristicDetector

enriched = inject_physical_residuals(df, rated_capacity_kw=240.0)
alerts = PhysicalHeuristicDetector().detect(enriched)
```

Or via CLI:

```bash
odse detect -i thokoza_telemetry_6mo.csv.gz --rated-kw 240 \
  --weather durban_weather_6mo.csv.gz -o thokoza_alerts.json
```

Next: commercial multi-step precursor forecasting stays in the platform world-model stack; open-source ODS-E owns first-line physical guardrails (SEP-053).
